### Star Schema - Analytics ETL

**What this notebook does:**
```
PostgreSQL (public schema)          PostgreSQL (analytics schema)
───────────────────────────         ──────────────────────────────
blood_reports          ──ETL──►     fact_blood_tests
customers              ──ETL──►     dim_patient  (SCD Type 2)
wearable CSV in MinIO  ──ETL──►     fact_wearable_daily
                                    dim_date
                                    dim_test_type
                                    dim_device
```

**Run order:** Cells must run top to bottom - dimensions first, facts second.

## 1. Imports & Connections

In [ ]:
!pip install psycopg2-binary python-dotenv minio pandas --quiet

In [ ]:
import os
import io
import psycopg2
import pandas as pd
from datetime import date, timedelta
from minio import Minio
from dotenv import load_dotenv

load_dotenv()

def get_pg():
    return psycopg2.connect(
        host="localhost", port=5432,
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD")
    )

def get_minio():
    return Minio(
        "localhost:9000",
        access_key=os.getenv("MINIO_ROOT_USER"),
        secret_key=os.getenv("MINIO_ROOT_PASSWORD"),
        secure=False
    )

pg    = get_pg()
minio = get_minio()
print("✅ Connected")

---
## 2. Create Analytics Schema & All Tables
Run once. Creates a separate `analytics` schema so your existing `public` tables are untouched.

In [ ]:
DDL = """

-- separate schema for analytics
CREATE SCHEMA IF NOT EXISTS analytics;

-- ── DIMENSION: DATE ─────────────────────────────────────────────────
-- Pre-expanded calendar table. Generated once, never changes.
CREATE TABLE IF NOT EXISTS analytics.dim_date (
    date_key    INT PRIMARY KEY,          -- e.g. 20260514
    full_date   DATE UNIQUE NOT NULL,
    day         SMALLINT,
    month       SMALLINT,
    month_name  TEXT,
    quarter     SMALLINT,
    year        SMALLINT,
    is_weekend  BOOLEAN
);

-- ── DIMENSION: PATIENT (SCD Type 2) ─────────────────────────────────
-- One row per version of a patient record.
-- When patient details change: old row expires, new row inserted.
CREATE TABLE IF NOT EXISTS analytics.dim_patient (
    patient_key INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id TEXT NOT NULL,            -- natural key from public.customers
    name        TEXT,
    age         INT,
    gender      TEXT,
    blood_type  TEXT,
    city        TEXT,
    -- SCD Type 2 tracking columns
    valid_from  DATE NOT NULL DEFAULT CURRENT_DATE,
    valid_to    DATE,                     -- NULL = currently active
    is_current  BOOLEAN DEFAULT TRUE
);

-- ── DIMENSION: TEST TYPE ─────────────────────────────────────────────
-- Lookup table for blood test categories.
CREATE TABLE IF NOT EXISTS analytics.dim_test_type (
    test_type_key INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    test_name     TEXT UNIQUE NOT NULL,   -- e.g. Blood Panel, Lipid Panel
    category      TEXT                    -- e.g. Hematology, Chemistry
);

-- ── DIMENSION: DEVICE ────────────────────────────────────────────────
-- Lookup table for wearable devices.
CREATE TABLE IF NOT EXISTS analytics.dim_device (
    device_key   INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    device_name  TEXT UNIQUE NOT NULL,    -- e.g. Fitbit Charge 5
    device_type  TEXT,                    -- e.g. Wearable
    manufacturer TEXT
);

-- ── FACT: BLOOD TESTS ────────────────────────────────────────────────
-- One row per blood test report per patient.
CREATE TABLE IF NOT EXISTS analytics.fact_blood_tests (
    id                SERIAL PRIMARY KEY,
    patient_key       INT REFERENCES analytics.dim_patient(patient_key),
    date_key          INT REFERENCES analytics.dim_date(date_key),
    test_type_key     INT REFERENCES analytics.dim_test_type(test_type_key),
    -- measurements
    glucose           DECIMAL(6,2),
    hemoglobin        DECIMAL(5,2),
    cholesterol_total DECIMAL(6,2),
    cholesterol_hdl   DECIMAL(6,2),
    cholesterol_ldl   DECIMAL(6,2),
    triglycerides     DECIMAL(6,2),
    wbc               DECIMAL(6,2),
    rbc               DECIMAL(6,2),
    -- traceability back to raw table
    source_id         INT,               -- blood_reports.id
    UNIQUE (patient_key, date_key)        -- one report per patient per day
);

-- ── FACT: WEARABLE DAILY ─────────────────────────────────────────────
-- One row per patient per day of wearable data.
CREATE TABLE IF NOT EXISTS analytics.fact_wearable_daily (
    id              SERIAL PRIMARY KEY,
    patient_key     INT REFERENCES analytics.dim_patient(patient_key),
    date_key        INT REFERENCES analytics.dim_date(date_key),
    device_key      INT REFERENCES analytics.dim_device(device_key),
    -- measurements
    steps           INT,
    heart_rate_avg  DECIMAL(5,2),
    sleep_hours     DECIMAL(4,2),
    calories        INT,
    UNIQUE (patient_key, date_key)        -- one summary row per patient per day
);

-- ── FACT: MEDICAL IMAGES ─────────────────────────────────────────────
-- One row per medical image file (CT, X-Ray etc).
CREATE TABLE IF NOT EXISTS analytics.fact_medical_images (
    id           SERIAL PRIMARY KEY,
    patient_key  INT REFERENCES analytics.dim_patient(patient_key),
    date_key     INT REFERENCES analytics.dim_date(date_key),
    image_type   TEXT,                    -- CT | X-Ray | MRI
    file_size_mb DECIMAL(8,2),
    minio_path   TEXT UNIQUE              -- path in MinIO bucket
);

-- ── ETL CONTROL TABLE ────────────────────────────────────────────────
-- Tracks the last time each ETL job ran for incremental loading.
CREATE TABLE IF NOT EXISTS analytics.etl_control (
    job_name      TEXT PRIMARY KEY,
    last_loaded   TIMESTAMPTZ DEFAULT '2000-01-01'
);

-- register all ETL jobs
INSERT INTO analytics.etl_control (job_name) VALUES
    ('dim_patient'),
    ('fact_blood_tests'),
    ('fact_wearable_daily'),
    ('fact_medical_images')
ON CONFLICT (job_name) DO NOTHING;

"""

with pg.cursor() as cur:
    cur.execute(DDL)
pg.commit()
print("✅ Analytics schema and all tables created")

---
## 3. Populate dim_date
Generates one row for every calendar day from 2020 to 2035. Run once — never needs to run again.

In [ ]:
def populate_dim_date(pg, start="2020-01-01", end="2035-12-31"):
    """Generate one row per calendar day between start and end dates."""
    MONTH_NAMES = ["","January","February","March","April","May","June",
                   "July","August","September","October","November","December"]

    rows, d = [], date.fromisoformat(start)
    while d <= date.fromisoformat(end):
        rows.append((
            int(d.strftime("%Y%m%d")),   # date_key  e.g. 20260514
            d,                            # full_date
            d.day,                        # day
            d.month,                      # month
            MONTH_NAMES[d.month],         # month_name
            (d.month - 1) // 3 + 1,      # quarter  1-4
            d.year,                       # year
            d.weekday() >= 5              # is_weekend
        ))
        d += timedelta(days=1)

    with pg.cursor() as cur:
        cur.executemany("""
            INSERT INTO analytics.dim_date
                (date_key, full_date, day, month, month_name, quarter, year, is_weekend)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (date_key) DO NOTHING
        """, rows)
    pg.commit()
    print(f"✅ dim_date populated — {len(rows)} days ({start} → {end})")

populate_dim_date(pg)

---
## 4. Seed dim_test_type & dim_device
Static lookup tables — insert once, extend manually as needed.

In [ ]:
def seed_test_types(pg):
    """Insert known blood test types."""
    rows = [
        ("Blood Panel",  "Hematology"),
        ("Lipid Panel",  "Chemistry"),
        ("CBC",          "Hematology"),
        ("Metabolic",    "Chemistry"),
        ("Unknown",      "General"),
    ]
    with pg.cursor() as cur:
        cur.executemany("""
            INSERT INTO analytics.dim_test_type (test_name, category)
            VALUES (%s, %s)
            ON CONFLICT (test_name) DO NOTHING
        """, rows)
    pg.commit()
    print("✅ dim_test_type seeded")


def seed_devices(pg):
    """Insert known wearable devices."""
    rows = [
        ("Fitbit Charge 5", "Wearable", "Fitbit"),
        ("Apple Watch",     "Wearable", "Apple"),
        ("Garmin Vivosmart","Wearable", "Garmin"),
        ("Unknown Device",  "Wearable", "Unknown"),
    ]
    with pg.cursor() as cur:
        cur.executemany("""
            INSERT INTO analytics.dim_device (device_name, device_type, manufacturer)
            VALUES (%s, %s, %s)
            ON CONFLICT (device_name) DO NOTHING
        """, rows)
    pg.commit()
    print("✅ dim_device seeded")


seed_test_types(pg)
seed_devices(pg)

---
## 5. ETL — dim_patient (SCD Type 2)
Reads from `public.customers`. When a patient's details change, the old row expires and a new one is inserted.

In [ ]:
def load_dim_patient(pg):
    """
    SCD Type 2 load for dim_patient.
    - New customer   → insert new row
    - Changed data   → expire old row, insert new row
    - No change      → skip
    """
    with pg.cursor() as cur:

        # fetch all customers from operational table
        cur.execute("""
            SELECT customer_id, name, age, gender, blood_type, city
            FROM public.customers
        """)
        customers = cur.fetchall()

        new_count, updated_count = 0, 0

        for cust in customers:
            cid, name, age, gender, blood_type, city = cust

            # check if current active row already matches
            cur.execute("""
                SELECT patient_key, name, age, gender, blood_type, city
                FROM analytics.dim_patient
                WHERE customer_id = %s AND is_current = TRUE
            """, (cid,))
            existing = cur.fetchone()

            if not existing:
                # brand new customer — insert
                cur.execute("""
                    INSERT INTO analytics.dim_patient
                        (customer_id, name, age, gender, blood_type, city)
                    VALUES (%s,%s,%s,%s,%s,%s)
                """, (cid, name, age, gender, blood_type, city))
                new_count += 1

            elif existing[1:] != (name, age, gender, blood_type, city):
                # something changed — expire old row, insert new one (SCD Type 2)
                cur.execute("""
                    UPDATE analytics.dim_patient
                    SET valid_to = CURRENT_DATE, is_current = FALSE
                    WHERE patient_key = %s
                """, (existing[0],))
                cur.execute("""
                    INSERT INTO analytics.dim_patient
                        (customer_id, name, age, gender, blood_type, city)
                    VALUES (%s,%s,%s,%s,%s,%s)
                """, (cid, name, age, gender, blood_type, city))
                updated_count += 1
            # else: no change — skip silently

        # update ETL control timestamp
        cur.execute("""
            UPDATE analytics.etl_control
            SET last_loaded = NOW()
            WHERE job_name = 'dim_patient'
        """)

    pg.commit()
    print(f"✅ dim_patient — {new_count} new, {updated_count} updated")


load_dim_patient(pg)

---
## 6. ETL — fact_blood_tests
Reads from `public.blood_reports`. Only loads records newer than the last run (incremental).

In [ ]:
def load_fact_blood_tests(pg):
    """
    Incremental load — only picks up blood_reports rows
    added since the last time this job ran.
    """
    with pg.cursor() as cur:

        # get last loaded timestamp
        cur.execute("""
            SELECT last_loaded FROM analytics.etl_control
            WHERE job_name = 'fact_blood_tests'
        """)
        last_loaded = cur.fetchone()[0]
        print(f"  Loading records newer than: {last_loaded}")

        # fetch only new blood reports
        cur.execute("""
            SELECT id, customer_id, report_date,
                   glucose, hemoglobin, cholesterol_total,
                   cholesterol_hdl, cholesterol_ldl,
                   triglycerides, wbc, rbc
            FROM public.blood_reports
            WHERE indexed_at > %s
              AND report_date IS NOT NULL
        """, (last_loaded,))
        rows = cur.fetchall()
        print(f"  Found {len(rows)} new record(s)")

        inserted = 0
        for row in rows:
            src_id, cid, rdate, glucose, hb, chol, hdl, ldl, trig, wbc, rbc = row

            # look up surrogate keys
            cur.execute("""
                SELECT patient_key FROM analytics.dim_patient
                WHERE customer_id = %s AND is_current = TRUE
            """, (cid,))
            patient = cur.fetchone()

            cur.execute("""
                SELECT date_key FROM analytics.dim_date
                WHERE full_date = %s
            """, (rdate,))
            dk = cur.fetchone()

            cur.execute("""
                SELECT test_type_key FROM analytics.dim_test_type
                WHERE test_name = 'Unknown'
            """)
            ttk = cur.fetchone()

            if not patient or not dk:
                print(f"  ⚠️  Skipped source_id={src_id} — missing patient or date key")
                continue

            cur.execute("""
                INSERT INTO analytics.fact_blood_tests (
                    patient_key, date_key, test_type_key,
                    glucose, hemoglobin, cholesterol_total,
                    cholesterol_hdl, cholesterol_ldl,
                    triglycerides, wbc, rbc, source_id
                ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                ON CONFLICT (patient_key, date_key) DO NOTHING
            """, (
                patient[0], dk[0], ttk[0],
                glucose, hb, chol, hdl, ldl, trig, wbc, rbc, src_id
            ))
            inserted += cur.rowcount

        # update control timestamp
        cur.execute("""
            UPDATE analytics.etl_control
            SET last_loaded = NOW()
            WHERE job_name = 'fact_blood_tests'
        """)

    pg.commit()
    print(f"✅ fact_blood_tests — {inserted} inserted")


load_fact_blood_tests(pg)

---
## 7. ETL — fact_wearable_daily
Reads wearable CSV files directly from MinIO and loads daily summaries.

In [ ]:
BUCKET = "health-data"

def load_wearable_csv(minio, object_path):
    """Download a wearable CSV from MinIO and return as a DataFrame."""
    response = minio.get_object(BUCKET, object_path)
    return pd.read_csv(io.BytesIO(response.read()))


def load_fact_wearable(minio, pg):
    """
    Scan MinIO for wearable CSVs and load daily summaries.
    Skips files already loaded (ON CONFLICT DO NOTHING).
    """
    objects = minio.list_objects(BUCKET, prefix="customers/", recursive=True)
    csvs    = [o for o in objects if "wearables" in o.object_name
               and o.object_name.endswith(".csv")]

    print(f"  Found {len(csvs)} wearable CSV file(s)")
    inserted = 0

    with pg.cursor() as cur:
        for obj in csvs:
            customer_id = obj.object_name.split("/")[1]

            # get patient surrogate key
            cur.execute("""
                SELECT patient_key FROM analytics.dim_patient
                WHERE customer_id = %s AND is_current = TRUE
            """, (customer_id,))
            patient = cur.fetchone()
            if not patient:
                print(f"  ⚠️  No patient found for {customer_id} — skipping")
                continue

            # get default device key
            cur.execute("""
                SELECT device_key FROM analytics.dim_device
                WHERE device_name = 'Unknown Device'
            """)
            device_key = cur.fetchone()[0]

            df = load_wearable_csv(minio, obj.object_name)

            # expected CSV columns: date, steps, heart_rate, sleep_hours, calories
            # adjust column names to match your actual CSV
            for _, row in df.iterrows():
                cur.execute("""
                    SELECT date_key FROM analytics.dim_date
                    WHERE full_date = %s
                """, (row.get("date"),))
                dk = cur.fetchone()
                if not dk:
                    continue

                cur.execute("""
                    INSERT INTO analytics.fact_wearable_daily
                        (patient_key, date_key, device_key,
                         steps, heart_rate_avg, sleep_hours, calories)
                    VALUES (%s,%s,%s,%s,%s,%s,%s)
                    ON CONFLICT (patient_key, date_key) DO NOTHING
                """, (
                    patient[0], dk[0], device_key,
                    row.get("steps"),
                    row.get("heart_rate"),
                    row.get("sleep_hours"),
                    row.get("calories")
                ))
                inserted += cur.rowcount

        cur.execute("""
            UPDATE analytics.etl_control SET last_loaded = NOW()
            WHERE job_name = 'fact_wearable_daily'
        """)

    pg.commit()
    print(f"✅ fact_wearable_daily — {inserted} rows inserted")


load_fact_wearable(minio, pg)

---
## 8. ETL — fact_medical_images
Scans MinIO DICOM folder and records image metadata. No file download needed — just the object metadata.

In [ ]:
def load_fact_medical_images(minio, pg):
    """
    Scan MinIO DICOM folder and store image metadata.
    No file content is read — just path, type, and size.
    """
    objects = minio.list_objects(BUCKET, prefix="customers/", recursive=True)
    images  = [o for o in objects if "dicom" in o.object_name]

    print(f"  Found {len(images)} DICOM file(s)")
    inserted = 0

    with pg.cursor() as cur:
        for obj in images:
            parts       = obj.object_name.split("/")
            customer_id = parts[1]
            file_name   = parts[-1]

            # detect image type from filename  e.g. CT_02_... or XR_01_...
            prefix     = file_name.split("_")[0].upper()
            image_type = {"CT": "CT Scan", "XR": "X-Ray", "MR": "MRI"}.get(prefix, "Unknown")

            cur.execute("""
                SELECT patient_key FROM analytics.dim_patient
                WHERE customer_id = %s AND is_current = TRUE
            """, (customer_id,))
            patient = cur.fetchone()
            if not patient:
                continue

            # use today as date_key since DICOM files have no embedded date here
            today_key = int(date.today().strftime("%Y%m%d"))

            cur.execute("""
                INSERT INTO analytics.fact_medical_images
                    (patient_key, date_key, image_type, file_size_mb, minio_path)
                VALUES (%s,%s,%s,%s,%s)
                ON CONFLICT (minio_path) DO NOTHING
            """, (
                patient[0],
                today_key,
                image_type,
                round(obj.size / (1024 * 1024), 2),
                obj.object_name
            ))
            inserted += cur.rowcount

        cur.execute("""
            UPDATE analytics.etl_control SET last_loaded = NOW()
            WHERE job_name = 'fact_medical_images'
        """)

    pg.commit()
    print(f"✅ fact_medical_images — {inserted} rows inserted")


load_fact_medical_images(minio, pg)

---
## 9. Run Full ETL Pipeline
Run all ETL jobs in the correct order with one cell.
Safe to re-run anytime — incremental load means no duplicates ever.

In [ ]:
def run_full_etl():
    """Run all ETL jobs in dependency order."""
    pg_conn    = get_pg()
    minio_conn = get_minio()

    print("\n── Step 1: Patients ──────────────────")
    load_dim_patient(pg_conn)

    print("\n── Step 2: Blood Tests ───────────────")
    load_fact_blood_tests(pg_conn)

    print("\n── Step 3: Wearables ─────────────────")
    load_fact_wearable(minio_conn, pg_conn)

    print("\n── Step 4: Medical Images ────────────")
    load_fact_medical_images(minio_conn, pg_conn)

    pg_conn.close()
    print("\n🎉 Full ETL complete")


run_full_etl()

---
## 10. Verify — Check What's in the Analytics Schema

In [ ]:
def show_counts(pg):
    """Print row counts for all analytics tables."""
    tables = [
        "analytics.dim_patient",
        "analytics.dim_date",
        "analytics.dim_test_type",
        "analytics.dim_device",
        "analytics.fact_blood_tests",
        "analytics.fact_wearable_daily",
        "analytics.fact_medical_images",
    ]
    print("\n📊 Analytics table row counts:")
    with pg.cursor() as cur:
        for t in tables:
            cur.execute(f"SELECT COUNT(*) FROM {t}")
            count = cur.fetchone()[0]
            print(f"  {t:<40} {count:>6} rows")


def show_etl_log(pg):
    """Show last run time for each ETL job."""
    print("\n⏱️  ETL last run times:")
    with pg.cursor() as cur:
        cur.execute("SELECT job_name, last_loaded FROM analytics.etl_control ORDER BY job_name")
        for job, ts in cur.fetchall():
            print(f"  {job:<25} {ts}")


show_counts(pg)
show_etl_log(pg)

---
## 11. Sample Analytics Queries
These are the kinds of queries Grafana will run against the analytics schema.

In [ ]:
def run_query(pg, sql, label):
    with pg.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    print(f"\n📊 {label}")
    print("  " + " | ".join(cols))
    print("  " + "-" * 60)
    for row in rows:
        print("  " + " | ".join(str(v) for v in row))


# Average blood markers per patient
run_query(pg, """
    SELECT p.customer_id,
           ROUND(AVG(f.glucose),2)    AS avg_glucose,
           ROUND(AVG(f.hemoglobin),2) AS avg_hemoglobin
    FROM analytics.fact_blood_tests f
    JOIN analytics.dim_patient p ON f.patient_key = p.patient_key
    GROUP BY p.customer_id
    ORDER BY p.customer_id
""", "Avg Blood Markers per Patient")


# Monthly blood test counts
run_query(pg, """
    SELECT d.year, d.month_name, COUNT(*) AS tests
    FROM analytics.fact_blood_tests f
    JOIN analytics.dim_date d ON f.date_key = d.date_key
    GROUP BY d.year, d.month, d.month_name
    ORDER BY d.year, d.month
""", "Monthly Blood Test Volume")


# Image count by type
run_query(pg, """
    SELECT image_type, COUNT(*) AS total,
           ROUND(SUM(file_size_mb),2) AS total_mb
    FROM analytics.fact_medical_images
    GROUP BY image_type
""", "Medical Images by Type")